# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading and exploring the FAIR² colorectal cancer survivors dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get the dataset metadata
metadata = dataset.metadata
print(metadata.name + ': ' + metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# Find out the available record sets and the structure using Croissant API
record_sets = list(dataset.record_sets())
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    fields = rs['field'] if 'field' in rs else []
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields: {[f['@id'] for f in fields]}")
    print()

## 3. Data Extraction
Load data from all detected record sets into individual pandas DataFrames. All `@id` values are used to reference entities.

In [ ]:
# Prepare the list of record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]

# Dictionary to store pandas DataFrames for each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")

# For further exploration, pick the first non-empty record set
first_nonempty_rs = next((rid for rid, df in dataframes.items() if not df.empty), None)

if first_nonempty_rs is not None:
    print(f"\nFields in RecordSet '{first_nonempty_rs}':")
    print(dataframes[first_nonempty_rs].columns.tolist())
    display(dataframes[first_nonempty_rs].head())
else:
    print("No records found in any RecordSet.")

## 4. Exploratory Data Analysis (EDA)
We can filter, normalize, and group data using numeric/categorical fields. Below are example operations performed using `@id` fields.

In [ ]:
# Example: Filter and normalize a numeric field from the selected record set
import numpy as np

record_set_id = first_nonempty_rs
df = dataframes[record_set_id]

if not df.empty:
    # Try to find a numeric column based on typical schema field naming
    numeric_candidate_ids = [c for c in df.columns if df[c].dtype in [np.int64, np.float64, float, int] or pd.api.types.is_numeric_dtype(df[c])]
    if len(numeric_candidate_ids) == 0:
        print("No numeric fields detected in the record set for EDA.")
    else:
        numeric_field_id = numeric_candidate_ids[0]
        print(f"Selected numeric field (by @id): {numeric_field_id}")

        # Choose a threshold value for filtering
        threshold = df[numeric_field_id].quantile(0.8)  # e.g., top 20% as demo
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with '{numeric_field_id}' > {threshold:.2f} (total {len(filtered_df)}):")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a possible categorical/grouping field
        categorical_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
        if len(categorical_candidates) > 0:
            group_field_id = categorical_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
else:
    print("No data available for EDA in record set.")

## 5. Visualization
Visualize a numeric field distribution or the relationship between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_nonempty_rs is not None and not df.empty and len(numeric_candidate_ids) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group field exists, show boxplot
    if len(categorical_candidates) > 0:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we loaded a FAIR² Croissant-format dataset using the `mlcroissant` library, explored its metadata and schema using `@id` references, and extracted tabular data directly into dataframes via record set `@id`. We demonstrated EDA and visualizations using detected fields in the dataset.

You can further adapt the code for specific record sets and field IDs based on your data science needs and the dataset's Croissant schema. For more details on the dataset, see [FAIR² at SenScience](https://sen.science/doi/10.71728/senscience.qs2f-h81p).